<a href="https://colab.research.google.com/github/EstefaniaS5/Proyecto-Buscaminas---Grupo-1/blob/main/Avance_Proyecto_Buscaminas_Grupo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files
import re

FILAS = 16
COLS = 16


while True:
    try:
        N_MINAS = int(input(f"Ingrese el número de minas (1 - {FILAS*COLS - 1}): "))
        if 1 <= N_MINAS < FILAS * COLS:
            break
        else:
            print("Número fuera de rango.")
    except:
        print("Ingrese un número válido.")

def crear_tablero():
    return [[0] * COLS for _ in range(FILAS)]

def poner_minas(tablero):
    posiciones = random.sample(range(FILAS * COLS), N_MINAS)
    for pos in posiciones:
        fila = pos // COLS
        col = pos % COLS
        tablero[fila][col] = -1
    return tablero

def calcular_vecinos(tablero):
    for f in range(FILAS):
        for c in range(COLS):
            if tablero[f][c] == -1:
                continue
            count = 0
            for df in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if df == 0 and dc == 0:
                        continue
                    nf, nc = f + df, c + dc
                    if 0 <= nf < FILAS and 0 <= nc < COLS:
                        if tablero[nf][nc] == -1:
                            count += 1
            tablero[f][c] = count
    return tablero

def imprimir_tablero(tablero):
    simbolos = {-1: " * "}
    print("   " + " ".join(f"{c:2}" for c in range(COLS)))
    print("   " + "───" * COLS)
    for f, fila in enumerate(tablero):
        fila_str = []
        for val in fila:
            fila_str.append(simbolos.get(val, f"{val:2} "))
        print(f"{f:2}|" + "".join(fila_str))

def generar_bloque_marie(tablero):
    lineas = ["; ── Tablero Buscaminas 16×16 para MARIE ──"]
    for f in range(FILAS):
        for c in range(COLS):
            valor = tablero[f][c]
            if valor == -1:
                valor_marie = 4095
            else:
                valor_marie = valor
            etiqueta = f"C{f:02d}{c:02d}"
            lineas.append(f"{etiqueta}, DEC {valor_marie}")
    return "\n".join(lineas)

def guardar_png_tablero(tablero, nombre_archivo="tablero_buscaminas.png"):
    colores = {
        -1: "#000000",
        0: "#d9d9d9",
        1: "#4da6ff",
        2: "#5cd65c",
        3: "#ff4d4d",
        4: "#1f4e79",
        5: "#990000",
        6: "#33cccc",
        7: "#9933cc",
        8: "#ffffff"
    }

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.set_xlim(-1, COLS)
    ax.set_ylim(-1, FILAS)
    ax.set_aspect('equal')
    ax.axis('off')

    for f in range(FILAS):
        for c in range(COLS):
            val = tablero[f][c]
            rect = patches.Rectangle((c, FILAS - 1 - f), 1, 1, facecolor=colores[val], edgecolor="black", linewidth=1)
            ax.add_patch(rect)

            if val == -1:
                ax.text(c + 0.5, FILAS - 1 - f + 0.5, "*", ha='center', va='center', fontsize=12, color='white', fontweight='bold')
            elif val != 0:
                color_texto = "white" if val in [4, 5, 7] else "black"
                ax.text(c + 0.5, FILAS - 1 - f + 0.5, str(val), ha='center', va='center', fontsize=11, color=color_texto, fontweight='bold')

    for c in range(COLS):
        ax.text(c + 0.5, FILAS + 0.1, str(c), ha='center', va='bottom', fontsize=9, color='#333333', fontweight='bold')

    for f in range(FILAS):
        ax.text(-0.1, FILAS - 1 - f + 0.5, str(f), ha='right', va='center', fontsize=9, color='#333333', fontweight='bold')

    ax.text(COLS / 2, FILAS + 0.6, "Columnas", ha='center', va='bottom', fontsize=10, color='#555555')
    ax.text(-0.7, FILAS / 2, "Filas", ha='center', va='center', fontsize=10, color='#555555', rotation=90)

    plt.tight_layout()
    plt.savefig(nombre_archivo, dpi=200, bbox_inches='tight')
    plt.close()

def actualizar_archivo_marie_existente(tablero, ruta_archivo_original, ruta_archivo_salida):
    lineas_modificadas = []
    try:
        with open(ruta_archivo_original, 'r') as f_in:
            for linea in f_in:
                # Buscar líneas que contengan etiquetas CXXYY, DEC ZZZZ
                match = re.match(r'(C(\d{2})(\d{2})),\s*DEC\s*(\d+)', linea)
                if match:
                    etiqueta, s_fila, s_col, valor_antiguo = match.groups()
                    fila = int(s_fila)
                    col = int(s_col)

                    # Asegurarse de que las coordenadas estén dentro de los límites del tablero generado
                    if 0 <= fila < FILAS and 0 <= col < COLS:
                        nuevo_valor = tablero[fila][col]
                        valor_marie = 4095 if nuevo_valor == -1 else nuevo_valor
                        lineas_modificadas.append(f"{etiqueta}, DEC {valor_marie}\n")
                    else:
                        # Si las coordenadas están fuera del rango, mantener la línea original
                        lineas_modificadas.append(linea)
                else:
                    # Si la línea no coincide con el patrón, mantenerla como está
                    lineas_modificadas.append(linea)
    except FileNotFoundError:
        print(f"Error: El archivo original no fue encontrado en la ruta '{ruta_archivo_original}'")
        return None
    except Exception as e:
        print(f"Ocurrió un error al leer o procesar el archivo: {e}")
        return None

    try:
        with open(ruta_archivo_salida, 'w') as f_out:
            f_out.writelines(lineas_modificadas)
        print(f"Archivo '{ruta_archivo_salida}' generado correctamente con los valores actualizados.")
        return ruta_archivo_salida
    except Exception as e:
        print(f"Ocurrió un error al escribir el archivo de salida: {e}")
        return None

tablero = crear_tablero()
tablero = poner_minas(tablero)
tablero = calcular_vecinos(tablero)

print("=== TABLERO BUSCAMINAS 16×16 ===\n")
imprimir_tablero(tablero)

print("\n=== BLOQUE MARIE COMPLETO (256 celdas) ===\n")
bloque = generar_bloque_marie(tablero)
print(bloque)

with open("tablero_marie.mas", "w") as f:
    f.write(bloque)

guardar_png_tablero(tablero)

# Nueva lógica para actualizar el archivo .mas existente
ruta_archivo_original_marie = "/content/MARIE/Codigo MARIE.mas"
ruta_archivo_salida_marie_actualizado = "codigo_marie_actualizado.mas"

archivo_actualizado = actualizar_archivo_marie_existente(
    tablero, ruta_archivo_original_marie, ruta_archivo_salida_marie_actualizado
)

# files.download("tablero_marie.mas") # Línea eliminada
files.download("tablero_buscaminas.png")

if archivo_actualizado:
    files.download(archivo_actualizado)


Ingrese el número de minas (1 - 255): 40
=== TABLERO BUSCAMINAS 16×16 ===

    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15
   ────────────────────────────────────────────────
 0| 0  0  0  0  0  0  1  1  1  0  0  0  0  1  *  1 
 1| 0  0  0  1  1  1  1  *  2  2  1  2  1  3  2  2 
 2| 1  1  1  2  *  1  1  2  *  3  *  2  *  2  *  1 
 3| *  2  1  *  2  1  0  2  3  *  2  2  2  3  2  1 
 4| *  2  1  1  1  0  1  2  *  2  1  0  1  *  2  1 
 5| 1  1  0  0  0  0  1  *  2  1  0  0  1  3  *  2 
 6| 0  0  0  0  0  0  1  1  1  0  0  0  0  3  *  3 
 7| 0  1  1  1  0  0  0  0  0  0  0  0  0  3  *  3 
 8| 1  2  *  2  1  1  1  1  1  0  0  0  0  2  *  2 
 9| *  3  1  3  *  2  1  *  1  0  0  0  1  2  2  1 
10| *  4  1  2  *  2  1  1  1  0  0  0  1  *  2  1 
11| *  *  1  1  1  1  0  1  1  1  1  1  3  2  3  * 
12| 2  2  2  1  2  2  3  3  *  1  1  *  2  *  2  1 
13| 1  2  3  *  2  *  *  *  2  1  1  2  3  2  2  1 
14| 1  *  *  3  3  3  3  2  1  0  0  1  *  1  1  * 
15| 1  2  2  2  *  1  0  0  0  0  0  1  1 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>